# Transfer_Learning_With__Conv_NeXt_Pre-trained_on_ImageNet

## Imports and Device Setup

In [ ]:
import torch
import glob
import matplotlib.pylab as plt
from torch.utils.data import Dataset
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision.models import resnet50, ResNet50_Weights 
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# See which device is being used
print(device)

import torch.optim as optim
from torch.optim.lr_scheduler import ExponentialLR

# Path Setup

In [ ]:
TRAIN_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Train"
VAL_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Val"
TEST_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Test"

# Transforms, Normalization and DataLoaders

In [ ]:
# Transforms 
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])



test_transform = transforms.Compose([transforms.Resize((224,224)),\
    transforms.ToTensor(),transforms.Normalize(mean=[0.485, 0.456, 0.406] ,std=[0.229, 0.224, 0.225])])


# Load datasets
train_dataset = ImageFolder(root=TRAIN_PATH, transform= train_transform)
val_dataset = ImageFolder(root=VAL_PATH, transform= val_transform)
test_dataset = ImageFolder(root=TEST_PATH, transform= test_transform)

Frozen_batch_size = 32
num_workers = 2

# DataLoaders
trainloader = DataLoader(train_dataset,batch_size=Frozen_batch_size,shuffle=True,num_workers=num_workers,pin_memory=True)
valloader = DataLoader(val_dataset, batch_size=Frozen_batch_size, shuffle=False, num_workers=num_workers)
testloader = DataLoader(test_dataset, batch_size=Frozen_batch_size, shuffle=False, num_workers=num_workers)

# Model Setup

In [ ]:
class GarbageModel(nn.Module):
    def __init__(self, num_classes, input_shape, transfer=False):
        super().__init__()

        weights = ResNet50_Weights.DEFAULT if transfer else None
        self.feature_extractor = resnet50(weights=weights)

        # Freeze backbone
        if transfer:
            for param in self.feature_extractor.parameters():
                param.requires_grad = False

        # Replace classifier
        n_features = self.feature_extractor.fc.in_features
        self.feature_extractor.fc = nn.Linear(n_features, num_classes)

    def forward(self, x):
        return self.feature_extractor(x)

# Model Instantiation

In [ ]:
model = GarbageModel(4, (3,224,224), True)
model.to(device)

# Loss Function, Optimizer & Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss() # Loss function
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),lr=1e-3)
scheduler = ExponentialLR(optimizer, gamma=0.9) #Gradually tunes down lr as model trains

# Train/Val Loop

In [ ]:
best_loss = float('inf')
nepochs = 5
PATH = '/home/le.song/Assignment2/ResNET_garbage_best_image_model_Frozen.pth'

for epoch in range(nepochs):
    #Training
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        #Resets Grad to avoid accumulation across batches
        optimizer.zero_grad()
        #Predict
        outputs = model(inputs)
        #Loss tracking
        loss = criterion(outputs, labels)
        #Gradient computation
        loss.backward()
        #Weights update
        optimizer.step()
        #accumulate
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss /= (i + 1)  # Train-loss
    train_acc = 100 * correct / total
    scheduler.step()

    # Validation mode, turning off Drop-out and BatchNorm
    model.eval()
    val_loss = 0.0
    # Almost same loop except no back-prop
    correct = 0
    total = 0
    with torch.no_grad():
        for i, (inputs, labels) in enumerate(valloader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        val_loss /= (i + 1) #Val-loss
        val_acc = 100 * correct / total

        print(f'Epoch {epoch+1}/{nepochs} | '
          f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.2f}%')

    # Save best model
    if val_loss < best_loss:
        print("Saving best model")
        torch.save(model.state_dict(), PATH)
        best_loss = val_loss

print("Model Saved to: ", PATH)